In [ ]:
%pip install -qU "langchain==0.3.27" "langchain-core<1.0.0,>=0.3.78" "langchain-text-splitters<1.0.0,>=0.3.9" langchain_ollama langchain_chroma langchain_community google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client python-docx

In [ ]:
import json
import logging
import os
import re
import sys
import hashlib
from pathlib import Path
from enum import StrEnum
from typing import Iterable, Optional

import gradio as gr
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError
from docx import Document as DocxDocument

In [ ]:
app_logger = logging.getLogger('drive_sage')
app_logger.setLevel(logging.DEBUG)

if not app_logger.handlers:
    stream_handler = logging.StreamHandler(sys.stdout)
    log_formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    stream_handler.setFormatter(log_formatter)
    app_logger.addHandler(stream_handler)

In [ ]:
DRIVE_SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
PROJECT_ROOT = Path.cwd()
CACHE_ROOT = PROJECT_ROOT / '.drive_sage'
CACHE_DOWNLOADS = CACHE_ROOT / 'downloads'
CACHE_VECTORSTORE = CACHE_ROOT / 'chroma'
OAUTH_TOKEN_PATH = CACHE_ROOT / 'token.json'
CACHE_MANIFEST_PATH = CACHE_ROOT / 'manifest.json'
OAUTH_CLIENT_SECRET = PROJECT_ROOT / 'client_secret_202216035337-4qson0c08g71u8uuihv6v46arv64nhvg.apps.googleusercontent.com.json'

for _p in (CACHE_ROOT, CACHE_DOWNLOADS, CACHE_VECTORSTORE):
    _p.mkdir(parents=True, exist_ok=True)

DOC_TYPE_CONFIG = {
    'txt': {
        'label': '.txt - Plain text',
        'extensions': ['.txt'],
        'mime_types': ['text/plain'],
    },
    'md': {
        'label': '.md - Markdown',
        'extensions': ['.md'],
        'mime_types': ['text/markdown', 'text/plain'],
    },
    'docx': {
        'label': '.docx - Word (OpenXML)',
        'extensions': ['.docx'],
        'mime_types': ['application/vnd.openxmlformats-officedocument.wordprocessingml.document'],
    },
    'doc': {
        'label': '.doc - Word 97-2003',
        'extensions': ['.doc'],
        'mime_types': ['application/msword', 'application/vnd.ms-word.document.macroenabled.12'],
    },
    'gdoc': {
        'label': 'Google Docs (exported)',
        'extensions': ['.docx'],
        'mime_types': ['application/vnd.google-apps.document'],
    },
}

DOC_LABEL_TO_KEY = {cfg['label']: key for key, cfg in DOC_TYPE_CONFIG.items()}
DEFAULT_DOC_TYPE_KEYS = ['txt', 'md', 'docx', 'doc', 'gdoc']
DEFAULT_DOC_TYPE_LABELS = [DOC_TYPE_CONFIG[key]['label'] for key in DEFAULT_DOC_TYPE_KEYS]

MIME_TO_EXT = {}
for _k, _cfg in DOC_TYPE_CONFIG.items():
    _ext = _cfg['extensions'][0]
    for _mime in _cfg['mime_types']:
        MIME_TO_EXT[_mime] = _ext

GOOGLE_EXPORT_MIMES = {
    'application/vnd.google-apps.document': (
        'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
        '.docx'
    ),
}

MAX_DISTANCE = float(os.getenv('DRIVE_SAGE_DISTANCE_MAX', '1.2'))
MAX_SNIPPET_CHARS = 1200
SHA1_BLOCK_SIZE = 65536
EMBEDDING_MODEL_NAME = os.getenv('DRIVE_SAGE_EMBED_MODEL', 'nomic-embed-text')
CHAT_MODEL_NAME = os.getenv('DRIVE_SAGE_CHAT_MODEL', 'llama3.1:latest')

UI_CSS = """
#chat-column {
    height: 80vh;
}
#chat-column > div {
    height: 100%;
}
#chat-column .gradio-chatbot,
#chat-column .gradio-chat-interface,
#chat-column .gradio-chatinterface {
    height: 100%;
}
#chat-output {
    height: 100%;
}
#chat-output .overflow-y-auto {
    max-height: 100% !important;
}
#chat-output .h-full {
    height: 100% !important;
}
"""

In [ ]:
def create_drive_service():
    oauth_creds = None
    if OAUTH_TOKEN_PATH.exists():
        try:
            oauth_creds = Credentials.from_authorized_user_file(str(OAUTH_TOKEN_PATH), DRIVE_SCOPES)
        except Exception as load_exc:
            app_logger.warning('Failed to load cached credentials: %s', load_exc)
            OAUTH_TOKEN_PATH.unlink(missing_ok=True)
            oauth_creds = None

    if not oauth_creds or not oauth_creds.valid:
        if oauth_creds and oauth_creds.expired and oauth_creds.refresh_token:
            try:
                oauth_creds.refresh(Request())
            except Exception as refresh_exc:
                app_logger.warning('Refreshing credentials failed: %s', refresh_exc)
                oauth_creds = None

        if not oauth_creds or not oauth_creds.valid:
            if not OAUTH_CLIENT_SECRET.exists():
                raise FileNotFoundError(
                    'client_secret.json not found. Download it from Google Cloud Console and place it next to this notebook.'
                )
            oauth_flow = InstalledAppFlow.from_client_secrets_file(str(OAUTH_CLIENT_SECRET), DRIVE_SCOPES)
            oauth_creds = oauth_flow.run_local_server(port=0)

        with OAUTH_TOKEN_PATH.open('w', encoding='utf-8') as token_fp:
            token_fp.write(oauth_creds.to_json())

    return build('drive', 'v3', credentials=oauth_creds)

In [ ]:
def read_manifest() -> dict:
    if CACHE_MANIFEST_PATH.exists():
        try:
            with CACHE_MANIFEST_PATH.open('r', encoding='utf-8') as fp:
                parsed = json.load(fp)
            if isinstance(parsed, dict):
                normalized: dict[str, dict] = {}
                for doc_id, entry in parsed.items():
                    if isinstance(entry, dict):
                        normalized[doc_id] = entry
                    else:
                        normalized[doc_id] = {'modified': str(entry)}
                return normalized
        except json.JSONDecodeError:
            app_logger.warning('Manifest file is corrupted; resetting cache.')
    return {}

def write_manifest(manifest: dict) -> None:
    with CACHE_MANIFEST_PATH.open('w', encoding='utf-8') as fp:
        json.dump(manifest, fp, indent=2)

In [ ]:
class MetaFields(StrEnum):
    ID = 'id'
    SOURCE = 'source'
    PARENT_ID = 'parent_id'
    FILE_TYPE = 'file_type'
    TITLE = 'title'
    MODIFIED = 'modified'

def meta_key(key: MetaFields) -> str:
    return key.value

embedder = OllamaEmbeddings(model=EMBEDDING_MODEL_NAME)

try:
    chroma_store = Chroma(
        collection_name='drive_sage',
        embedding_function=embedder,
    )
except Exception as init_exc:
    app_logger.exception('Failed to initialize in-memory Chroma vector store')
    raise RuntimeError('Unable to initialize Chroma vector store without persistence.') from init_exc

kv_docstore = InMemoryStore()
chat_llm = ChatOllama(model=CHAT_MODEL_NAME)

TEXT_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=['\n\n', '\n', ' ', '']
)
MD_HEADERS = [('#', 'Header 1'), ('##', 'Header 2'), ('###', 'Header 3')]
MD_SPLITTER = MarkdownHeaderTextSplitter(headers_to_split_on=MD_HEADERS, strip_headers=False)

In [ ]:
def normalize_filename(name: str, max_length: int = 120) -> str:
    cleaned = re.sub(r'[^A-Za-z0-9._-]', '_', name)
    cleaned = cleaned.strip('._') or 'untitled'
    return cleaned[:max_length]

def guess_extension(meta: dict) -> str:
    mime = meta.get('mimeType', '')
    fname = meta.get('name')
    if fname and Path(fname).suffix:
        return Path(fname).suffix.lower()
    if mime in GOOGLE_EXPORT_MIMES:
        return GOOGLE_EXPORT_MIMES[mime][1]
    return MIME_TO_EXT.get(mime, '.txt')

def get_cached_path(meta: dict) -> Path:
    doc_id = meta.get('id', 'unknown')
    ext = guess_extension(meta)
    safe_stem = normalize_filename(Path(meta.get('name', doc_id)).stem)
    return CACHE_DOWNLOADS / f'{safe_stem}_{doc_id}{ext}'

def sha1sum(path: Path) -> str:
    h = hashlib.sha1()
    with path.open('rb') as fh:
        while True:
            block = fh.read(SHA1_BLOCK_SIZE)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def manifest_rev(entry: dict | str | None) -> Optional[str]:
    if entry is None:
        return None
    if isinstance(entry, str):
        return entry
    if isinstance(entry, dict):
        return entry.get('modified')
    return None

def upsert_manifest(manifest: dict, *, file_id: str, modified: str, path: Path, mime_type: str, name: str) -> None:
    manifest[file_id] = {
        'modified': modified,
        'path': str(path),
        'mimeType': mime_type,
        'name': name,
        'file_type': Path(path).suffix.lower(),
    }

In [ ]:
def list_drive_candidates(service, folder_id: Optional[str], allowed_mimes: list[str], limit: Optional[int]) -> list[dict]:
    clauses = ["trashed = false"]
    mimes = allowed_mimes or list(MIME_TO_EXT.keys())
    mime_clause = ' or '.join([f"mimeType = '{m}'" for m in mimes])
    clauses.append(f'({mime_clause})')
    if folder_id:
        clauses.append(f"'{folder_id}' in parents")
    q = ' and '.join(clauses)

    items: list[dict] = []
    token: Optional[str] = None

    while True:
        page_size = min(100, limit - len(items)) if limit else 100
        if page_size <= 0:
            break
        try:
            resp = service.files().list(
                q=q,
                spaces='drive',
                fields='nextPageToken, files(id, name, mimeType, modifiedTime)',
                orderBy='modifiedTime desc',
                pageToken=token,
                pageSize=page_size,
            ).execute()
        except HttpError as http_exc:
            raise RuntimeError(f'Google Drive API error: {http_exc}') from http_exc

        batch = resp.get('files', [])
        items.extend(batch)
        if limit and len(items) >= limit:
            return items[:limit]
        token = resp.get('nextPageToken')
        if not token:
            break
    return items

def fetch_drive_file(service, meta: dict, manifest: dict) -> Path:
    file_id = meta['id']
    mime = meta.get('mimeType', '')
    target_path = get_cached_path(meta)
    export_mime = None

    if mime in GOOGLE_EXPORT_MIMES:
        export_mime, export_ext = GOOGLE_EXPORT_MIMES[mime]
        if target_path.suffix.lower() != export_ext:
            target_path = target_path.with_suffix(export_ext)

    req = (
        service.files().export_media(fileId=file_id, mimeType=export_mime)
        if export_mime
        else service.files().get_media(fileId=file_id)
    )

    app_logger.debug('Downloading %s (%s) -> %s', meta.get('name', file_id), file_id, target_path)
    with target_path.open('wb') as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            status, done = dl.next_chunk()
            if status:
                app_logger.debug('Download progress %.0f%%', status.progress() * 100)

    upsert_manifest(
        manifest,
        file_id=file_id,
        modified=meta.get('modifiedTime', ''),
        path=target_path,
        mime_type=mime,
        name=meta.get('name', target_path.name),
    )
    return target_path

In [ ]:
def docx_to_text(path: Path) -> str:
    docx_obj = DocxDocument(str(path))
    lines = [p.text.strip() for p in docx_obj.paragraphs if p.text.strip()]
    return '\n'.join(lines)

def read_docs(
    path: Path,
    *,
    source_id: Optional[str] = None,
    file_type: Optional[str] = None,
    modified: Optional[str] = None,
    display_name: Optional[str] = None,
 ) -> list[Document]:
    suffix = (file_type or path.suffix or '.txt').lower()
    try:
        if suffix in {'.txt', '.md'}:
            txt_loader = TextLoader(str(path), encoding='utf-8')
            docs = txt_loader.load()
        elif suffix == '.docx':
            docs = [Document(page_content=docx_to_text(path), metadata={'source': str(path)})]
        else:
            raise ValueError(f'Unsupported file type: {suffix}')
    except UnicodeDecodeError as decode_exc:
        raise ValueError(f'Failed to read {path}: {decode_exc}') from decode_exc

    base_meta = {
        meta_key(MetaFields.SOURCE): str(path),
        meta_key(MetaFields.FILE_TYPE): suffix,
        meta_key(MetaFields.TITLE): display_name or path.name,
    }
    if source_id:
        base_meta[meta_key(MetaFields.ID)] = source_id
    if modified:
        base_meta[meta_key(MetaFields.MODIFIED)] = modified

    out: list[Document] = []
    for d in docs:
        content = d.page_content.strip()
        if not content:
            continue
        d.page_content = content
        d.metadata = {**d.metadata, **base_meta}
        out.append(d)
    return out

def keep_nonempty(docs: Iterable[Document]) -> list[Document]:
    return [d for d in docs if d.page_content]

def split_into_chunks(doc: Document) -> list[Document]:
    parent = doc.metadata.get(meta_key(MetaFields.ID))
    if not parent:
        raise ValueError('Document is missing a stable identifier for chunking.')

    if doc.metadata.get(meta_key(MetaFields.FILE_TYPE)) == '.md':
        md_sections = MD_SPLITTER.split_text(doc.page_content)
        seeds = [
            Document(page_content=s.page_content, metadata={**doc.metadata, **s.metadata})
            for s in md_sections
        ]
    else:
        seeds = [doc]

    chunks = TEXT_SPLITTER.split_documents(seeds)
    for i, ch in enumerate(chunks):
        ch.metadata[meta_key(MetaFields.PARENT_ID)] = parent
        ch.metadata[meta_key(MetaFields.ID)] = f'{parent}::chunk-{i:04d}'
        ch.metadata.setdefault(meta_key(MetaFields.SOURCE), doc.metadata.get(meta_key(MetaFields.SOURCE)))
        ch.metadata.setdefault(meta_key(MetaFields.TITLE), doc.metadata.get(meta_key(MetaFields.TITLE)))
    return chunks

In [ ]:
def sync_and_index(folder_id=None, selected_types=None, file_limit=None, _state: bool = False, progress=gr.Progress(track_tqdm=False)):
    folder = (folder_id or '').strip() or None

    selection_labels = selected_types if selected_types is not None else DEFAULT_DOC_TYPE_LABELS
    if not isinstance(selection_labels, (list, tuple)):
        selection_labels = [selection_labels]
    selection_labels = list(selection_labels)

    if len(selection_labels) == 0:
        yield 'Select at least one file type before syncing.', False
        return

    chosen: list[str] = []
    for item in selection_labels:
        key = DOC_LABEL_TO_KEY.get(item, item)
        if key in DOC_TYPE_CONFIG:
            chosen.append(key)

    if not chosen:
        yield 'Select at least one file type before syncing.', False
        return

    allowed_mimes = sorted({m for k in chosen for m in DOC_TYPE_CONFIG[k]['mime_types']})

    limit_num: Optional[int] = None
    limit_warn: Optional[str] = None
    if file_limit not in (None, '', 0):
        try:
            parsed = int(file_limit)
            if parsed > 0:
                limit_num = parsed
            else:
                raise ValueError
        except (TypeError, ValueError):
            limit_warn = 'File limit must be a positive integer. Syncing all matching files instead.'

    messages: list[str] = []

    def push(msg: str) -> str:
        messages.append(msg)
        return '\n'.join(messages)

    if limit_warn:
        app_logger.warning(limit_warn)
        yield push(limit_warn), False

    progress(0, 'Authorizing Google Drive access...')
    yield push('Authorizing Google Drive access...'), False

    try:
        service = create_drive_service()
    except FileNotFoundError as missing_exc:
        err = f'Error: {missing_exc}'
        app_logger.error(err)
        yield push(err), False
        return
    except Exception as auth_exc:
        app_logger.exception('Drive authorization failed')
        err = f'Error authenticating with Google Drive: {auth_exc}'
        yield push(err), False
        return

    list_msg = 'Listing documents' + (f' (limit {limit_num})' if limit_num else '') + '...'
    progress(0, list_msg)
    yield push(list_msg), False

    try:
        files = list_drive_candidates(service, folder, allowed_mimes, limit_num)
    except Exception as list_exc:
        app_logger.exception('Listing Drive files failed')
        err = f'Error listing Google Drive files: {list_exc}'
        yield push(err), False
        return

    total = len(files)
    if total == 0:
        info = 'No documents matching the selected types were found in Google Drive.'
        yield push(info), True
        return

    manifest = read_manifest()
    downloaded = 0

    for idx, meta in enumerate(files, start=1):
        file_id = meta['id']
        name = meta.get('name', file_id)
        remote_mod = meta.get('modifiedTime', '')
        entry = manifest.get(file_id)
        cache_path = get_cached_path(meta)
        if isinstance(entry, dict) and entry.get('path'):
            cache_path = Path(entry['path'])
        cached_mod = manifest_rev(entry)

        if cached_mod == remote_mod and cache_path.exists():
            msg = f"{idx}/{total} Skipping cached file: {name} -> {cache_path}"
            progress(idx / total, msg)
            yield push(msg), False
            continue

        dl_msg = f"{idx}/{total} Downloading {name} -> {cache_path}"
        progress(max((idx - 0.5) / total, 0), dl_msg)
        yield push(dl_msg), False

        try:
            local_path = fetch_drive_file(service, meta, manifest)
            index_msg = f"{idx}/{total} Indexing {local_path.name}"
            progress(idx / total, index_msg)
            yield push(index_msg), False
            index_single_document(
                local_path,
                source_id=file_id,
                file_type=local_path.suffix,
                modified=remote_mod,
                display_name=name,
                manifest=manifest,
            )
            downloaded += 1
        except Exception as sync_exc:
            err = f"{idx}/{total} Failed to sync {name}: {sync_exc}"
            app_logger.exception(err)
            progress(idx / total, err)
            yield push(err), False

    if downloaded > 0:
        write_manifest(manifest)
        summary = f'Indexed {downloaded} new document(s) from Google Drive.'
    else:
        summary = 'Google Drive is already in sync.'

    progress(1, summary)
    yield push(summary), True

## RAG Pipeline

In [ ]:
def persist_store(_store) -> None:
    """In-memory mode: Chroma client does not persist between sessions."""
    return


def index_single_document(
    file_path: Path | str,
    *,
    source_id: Optional[str] = None,
    file_type: Optional[str] = None,
    modified: Optional[str] = None,
    display_name: Optional[str] = None,
    manifest: Optional[dict] = None,
 ) -> tuple[str, int]:
    path = Path(file_path).expanduser().resolve()
    stable_id = source_id or f'local::{sha1sum(path)}'

    docs = read_docs(
        path,
        source_id=stable_id,
        file_type=file_type,
        modified=modified,
        display_name=display_name,
    )
    docs = keep_nonempty(docs)
    if not docs:
        app_logger.warning('No readable content found in %s; skipping.', path)
        return stable_id, 0

    total_chunks = 0
    for d in docs:
        doc_id = d.metadata.get(meta_key(MetaFields.ID), stable_id)
        d.metadata[meta_key(MetaFields.ID)] = doc_id

        chroma_store.delete(where={meta_key(MetaFields.PARENT_ID): doc_id})
        chunks = split_into_chunks(d)
        if not chunks:
            continue

        chroma_store.add_documents(chunks)
        kv_docstore.mset([(doc_id, d)])
        total_chunks += len(chunks)

    persist_store(chroma_store)
    if manifest is not None and not source_id:
        upsert_manifest(
            manifest,
            file_id=stable_id,
            modified=sha1sum(path),
            path=path,
            mime_type=file_type or Path(path).suffix or '.txt',
            name=display_name or path.name,
        )
    return stable_id, total_chunks

### LLM Interaction

In [ ]:
def fetch_context(query: str, *, top_k: int = 8, distance_threshold: Optional[float] = MAX_DISTANCE):
    results = chroma_store.similarity_search_with_score(query, k=top_k)
    app_logger.info(f'Matching records: {len(results)}')

    kept: list[tuple[Document, float]] = []
    for doc, score in results:
        if score is None:
            continue
        dist = float(score)
        print(f'DEBUG: Retrieved doc source={doc.metadata.get(meta_key(MetaFields.SOURCE))} distance={dist}')
        if distance_threshold is not None and dist > distance_threshold:
            app_logger.debug(
                'Skipping %s with distance %.4f (above threshold %.4f)',
                doc.metadata.get(meta_key(MetaFields.SOURCE)),
                dist,
                distance_threshold,
            )
            continue
        kept.append((doc, dist))

    if not kept:
        return []

    for doc, dist in kept:
        parent = doc.metadata.get(meta_key(MetaFields.PARENT_ID))
        if parent:
            parent_doc = kv_docstore.mget([parent])[0]
            if parent_doc and parent_doc.page_content:
                app_logger.debug(
                    'Parent preview (%s | %.3f): %s',
                    doc.metadata.get(meta_key(MetaFields.SOURCE), 'unknown'),
                    dist,
                    parent_doc.page_content[:400].replace('\n', ' '),
                )

    return kept


def format_context_sections(docs_with_scores: list[tuple[Document, float]]) -> str:
    blocks: list[str] = []
    for i, (doc, dist) in enumerate(docs_with_scores, start=1):
        src = doc.metadata.get(meta_key(MetaFields.SOURCE), 'unknown')
        snippet = doc.page_content.strip()[:MAX_SNIPPET_CHARS]
        blocks.append(
            f'[{i}] Source: {src}\n'
            f'Distance: {dist:.3f}\n'
            f'Content:\n{snippet}'
        )
    return '\n\n'.join(blocks)


def answer_stream(message, history):
    docs = fetch_context(message)
    if not docs:
        yield "I don't have enough information in the synced documents to answer that yet. Please sync additional files or adjust the filters."
        return

    ctx = format_context_sections(docs)
    system_prompt = f'''
    You are a retrieval-augmented assistant. Use ONLY the facts provided in the context to answer the user.
    If the context does not contain the answer, reply exactly: "I don't have enough information in the synced documents to answer that yet. Please sync additional files."

    Context:\n{ctx}
    '''

    msgs = [
        ('system', system_prompt),
        ('user', message)
    ]

    streaming = chat_llm.stream(msgs)
    out = ''
    for chunk in streaming:
        out += chunk.content or ''
        if not out:
            continue
        yield out

## Gradio UI

In [ ]:
def ui_chat(message, history, sync_ready):
    if message is None:
        return ''

    user_text = message.get('text', '')
    uploaded_files = message.get('files', [])
    latest_upload = Path(uploaded_files[-1]) if uploaded_files else None

    if latest_upload:
        manifest = read_manifest()
        doc_id, chunk_count = index_single_document(
            latest_upload,
            file_type=latest_upload.suffix,
            display_name=latest_upload.name,
            manifest=manifest,
        )
        write_manifest(manifest)
        app_logger.info('Indexed upload %s as %s with %s chunk(s)', latest_upload, doc_id, chunk_count)
        if not user_text:
            yield f'Indexed document from upload ({chunk_count} chunk(s)).'
            return

    if not user_text:
        return ''

    if not sync_ready and not uploaded_files:
        yield 'Sync Google Drive before chatting or upload a document first.'
        return

    for chunk in answer_stream(user_text, history):
        yield chunk

app_title = "Drive Sage"
with gr.Blocks(title=app_title, fill_height=True, css=UI_CSS) as app:
    gr.Markdown(f'# {app_title}')
    gr.Markdown('## Search your Google Drive knowledge base with fully local processing.')
    sync_state = gr.State(False)

    with gr.Row():
        with gr.Column(scale=3, elem_id='chat-column'):
            gr.ChatInterface(
                fn=ui_chat,
                chatbot=gr.Chatbot(height='80vh', elem_id='chat-output'),
                type='messages',
                textbox=gr.MultimodalTextbox(
                    file_types=['text', '.txt', '.md'],
                    autofocus=True,
                    elem_id='chat-input',
                ),
                additional_inputs=[sync_state],
            )
        with gr.Column(scale=2, min_width=320):
            gr.Markdown('### Google Drive Sync')
            drive_folder = gr.Textbox(
                label='Folder ID (optional)',
                placeholder='Leave blank to scan My Drive root',
            )
            file_types = gr.CheckboxGroup(
                label='File types to sync',
                choices=[cfg['label'] for cfg in DOC_TYPE_CONFIG.values()],
                value=DEFAULT_DOC_TYPE_LABELS,
            )
            file_limit = gr.Number(
                label='Max files to sync (leave blank for all)',
                value=20,
            )
            sync_button = gr.Button('Sync Google Drive')
            sync_status = gr.Markdown('No sync performed yet.')

            sync_button.click(
                sync_and_index,
                inputs=[drive_folder, file_types, file_limit, sync_state],
                outputs=[sync_status, sync_state],
            )

app.launch(debug=True)